# Multi-task Swin-based Classifier with Training Curves and Grad-CAM Visualizer

This notebook trains a Swin-based classifier for lung CT images and provides:
- 30 epochs training with **early stopping**
- Save the best model to `best_model.pth`
- Plots: training/validation loss & accuracy vs epochs
- Final cell: **loads the saved model** and displays **10 random examples** (GT, prediction, Grad-CAM overlay)

Notes:
- Dataset assumed at `./Dataset/<ClassName>/*.jpg` (adjust `DATA_DIR` if needed)
- Uses `timm` for model backbone. Install with `pip install timm` if missing.
- Set `NUM_WORKERS = 0` for Windows/Jupyter compatibility.
- The final visualization cell **always** loads the saved model and picks random images each run.

In [ ]:
# --- Imports ---
import os, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('torch:', torch.__version__)
print('timm:', timm.__version__)


In [ ]:
# --- User config ---
DATA_DIR = './Dataset'   # change if your dataset path is different
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0   # safe for Windows / Jupyter
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
LR = 3e-4
SEED = 42
BACKBONE = 'swin_small_patch4_window7_224'
PRETRAINED = True
SAVED_MODEL_PATH = 'best_model.pth'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# --- Build file list and labels ---
class_names = [
    'Benign',
    'Normal',
    'Adenocarcinoma',
    'Large_cell_carcinoma',
    'Squamous_Cell_Carcinoma'
]
all_files = []
for cls in class_names:
    cls_dir = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_dir):
        print(f'Warning: missing folder for class: {cls_dir}')
        continue
    for p in Path(cls_dir).glob('*'):
        if p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff']:
            all_files.append((str(p), cls))

if len(all_files) == 0:
    raise RuntimeError('No images found. Check DATA_DIR and class folders.')

df = pd.DataFrame(all_files, columns=['path','class'])
df['binary'] = (df['class'] != 'Normal').astype(int)
disease_map = {c:i for i,c in enumerate([c for c in class_names if c!='Normal'])}
def to_multi_label(r):
    if r['class']=='Normal':
        return -1
    return disease_map[r['class']]
df['multi'] = df.apply(to_multi_label, axis=1)

print('Class counts:')
print(df['class'].value_counts())


In [ ]:
# --- Train / validation split ---
train_df, val_df = train_test_split(df, test_size=0.20, stratify=df['class'], random_state=SEED)
print('Train:', len(train_df), 'Val:', len(val_df))


In [ ]:
# --- Transforms and Dataset ---
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.08, contrast=0.08),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
class LungCTDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transforms is not None:
            img = self.transforms(img)
        bin_lbl = int(row['binary'])
        multi_lbl = int(row['multi'])
        return img, bin_lbl, multi_lbl, row['path']

train_ds = LungCTDataset(train_df, transforms=train_transforms)
val_ds = LungCTDataset(val_df, transforms=val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print('Batches — train:', len(train_loader), 'val:', len(val_loader))


In [ ]:
# --- Model: Swin backbone with two heads (binary + multi-class) ---
class MultiTaskSwin(nn.Module):
    def __init__(self, backbone_name=BACKBONE, pretrained=PRETRAINED, num_multi=4):
        super().__init__()
        # create backbone without head
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        # binary head
        self.binary_head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
        # multi-class head
        self.multi_head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_multi)
        )
    def forward(self, x):
        feats = self.backbone(x)  # global pooled features
        out_bin = self.binary_head(feats)
        out_multi = self.multi_head(feats)
        return out_bin, out_multi

model = MultiTaskSwin().to(device)
print(model)


In [ ]:
# --- Losses, optimizer, scheduler ---
criterion_bin = nn.CrossEntropyLoss()
criterion_multi = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)


In [ ]:
# --- Training & Validation functions (with metrics) ---
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    all_targets = []
    all_preds = []
    for imgs, bin_lbls, multi_lbls, _ in loader:
        imgs = imgs.to(device)
        bin_lbls = bin_lbls.to(device)
        multi_lbls = multi_lbls.to(device)
        optimizer.zero_grad()
        out_bin, out_multi = model(imgs)
        loss_bin = criterion_bin(out_bin, bin_lbls)
        mask = (multi_lbls >= 0)
        if mask.any():
            loss_multi = criterion_multi(out_multi[mask], multi_lbls[mask])
        else:
            loss_multi = torch.tensor(0.0, device=device)
        loss = loss_bin + 0.8 * loss_multi
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = torch.argmax(out_bin.detach().cpu(), dim=1).numpy().tolist()
        all_preds.extend(preds)
        all_targets.extend(bin_lbls.detach().cpu().numpy().tolist())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    return epoch_loss, epoch_acc

def validate(model, loader, device):
    model.eval()
    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_multi_targets = []
    all_multi_preds = []
    with torch.no_grad():
        for imgs, bin_lbls, multi_lbls, _ in loader:
            imgs = imgs.to(device)
            bin_lbls = bin_lbls.to(device)
            multi_lbls = multi_lbls.to(device)
            out_bin, out_multi = model(imgs)
            loss_bin = criterion_bin(out_bin, bin_lbls)
            mask = (multi_lbls >= 0)
            if mask.any():
                loss_multi = criterion_multi(out_multi[mask], multi_lbls[mask])
            else:
                loss_multi = torch.tensor(0.0, device=device)
            loss = loss_bin + 0.8 * loss_multi
            running_loss += loss.item() * imgs.size(0)
            preds = torch.argmax(out_bin.detach().cpu(), dim=1).numpy().tolist()
            all_preds.extend(preds)
            all_targets.extend(bin_lbls.detach().cpu().numpy().tolist())
            if mask.any():
                pm = torch.argmax(out_multi[mask].detach().cpu(), dim=1).numpy().tolist()
                tm = multi_lbls[mask].detach().cpu().numpy().tolist()
                all_multi_preds.extend(pm)
                all_multi_targets.extend(tm)
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    multi_acc = None
    if len(all_multi_targets)>0:
        multi_acc = accuracy_score(all_multi_targets, all_multi_preds)
    return epoch_loss, epoch_acc, multi_acc


In [ ]:
# --- Training loop with early stopping and history tracking ---
best_val = -1.0
best_epoch = -1
patience = EARLY_STOPPING_PATIENCE
history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
start_time = time.time()
for epoch in range(MAX_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc, val_multi_acc = validate(model, val_loader, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch+1}/{MAX_EPOCHS} — train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f} | val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}, val_multi_acc: {val_multi_acc}")
    # scheduler on val acc
    scheduler.step(val_acc)
    # early stopping logic based on val_acc
    if val_acc > best_val:
        best_val = val_acc
        best_epoch = epoch
        torch.save(model.state_dict(), SAVED_MODEL_PATH)
        print('Saved best model at epoch', epoch+1)
        patience = EARLY_STOPPING_PATIENCE
    else:
        patience -= 1
        print('Patience left:', patience)
    if patience <= 0:
        print('Early stopping triggered')
        break
    t1 = time.time()
    print('Epoch time:', t1-t0)
total_time = time.time()-start_time
print('Training done. Best val acc:', best_val, 'at epoch', best_epoch+1)


In [ ]:
# --- Plot training curves (loss & accuracy) ---
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='train loss')
plt.plot(history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss vs Epochs')

plt.subplot(1,2,2)
plt.plot(history['train_acc'], label='train acc')
plt.plot(history['val_acc'], label='val acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy vs Epochs')

plt.tight_layout()
plt.show()

# Save history to CSV for later analysis
hist_df = pd.DataFrame(history)
hist_df.to_csv('training_history.csv', index=False)
print('Saved training_history.csv')


## Final visualization cell — loads saved model and shows 10 random examples with Grad-CAM

The cell below **loads `best_model.pth`** and selects 10 random images from the validation set, shows:
- original image (rescaled)
- ground truth label
- model prediction (binary + multi if applicable)
- Grad-CAM heatmap overlay

Run this cell any time after `best_model.pth` has been created. It will NOT retrain the model.

In [ ]:
# --- Utilities for Grad-CAM on backbone.feature maps ---
class GradCAM:
    def __init__(self, model, backbone_name=BACKBONE, device='cpu'):
        self.model = model
        self.device = device
        self.model.to(device)
        self.model.eval()
        # We'll attempt to hook into backbone.forward_features output by creating a wrapper
        self.activations = None
        self.gradients = None
        # register hooks to the backbone's feature extractor if possible
        # many timm models implement forward_features which returns a tensor before pooling
        backbone = self.model.backbone
        # We'll hook into the forward_features method by wrapping it
        # Create a handle to save activations and gradients
        def forward_features_hook(module, input, output):
            # output may be (B, C) for pooled or (B, C, H, W) for feature map
            self.activations = output.detach()
            return None
        # For gradients, we rely on registering hook on activations tensor dynamically in forward pass
        # Instead of registering module hooks, we'll call backbone.forward_features manually in compute
        
    def compute_cam(self, input_tensor, class_idx=None):
        """
        input_tensor: single image tensor (1,C,H,W) on device
        class_idx: index for binary head (0/1). We'll use binary prediction for saliency.
        returns: heatmap (H,W) normalized 0..1
        """
        # We'll perform a forward pass that captures features before pooling.
        backbone = self.model.backbone
        # Use backbone.forward_features if present to get spatial features for CAM
        # Many timm swin models compute forward_features and then pooling; we need the 2D map.
        activations = None
        grads = None
        # Custom forward to capture feature map and hook gradient
        def forward_and_hook(x):
            # call backbone.forward_features to get feature map if available
            if hasattr(backbone, 'forward_features'):
                feat_map = backbone.forward_features(x)
            else:
                # fallback: call backbone(x) and try to reshape - may produce pooled feats
                feat_map = backbone(x)
            return feat_map

        # Require gradient on input features
        input_tensor = input_tensor.to(self.device)
        # First get feature map
        # We'll run backbone.stage2 or forward_features; try to locate a spatial feat
        # Some backbones return (B, C) - in that case CAM will use gradients on that which is less spatial
        # Try to run a lower-level forward to retrieve spatial map using model.backbone.patch_embed or feature extractor
        # Simpler approach: run a forward hook on a likely submodule (like backbone.layers[-1])
        hook_handles = []
        saved = {'act': None, 'grad': None}
        # try to find a module with 4D output (convolutional block or stage)
        target_module = None
        for name, module in backbone.named_modules():
            # heuristically choose a module with 'layers' or 'stages' in name or last few modules
            pass
        # fallback choice: last module of backbone
        try:
            # Some backbones (swin) have 'layers' attribute
            cand = None
            if hasattr(backbone, 'layers'):
                cand = list(backbone.layers)[-1]
            elif hasattr(backbone, 'stages'):
                cand = list(backbone.stages)[-1]
            else:
                # last child
                cand = list(backbone.children())[-1]
            target_module = cand
        except Exception:
            target_module = None

        if target_module is None:
            # just run a normal forward and compute gradients on pooled features
            out_bin, out_multi = self.model(input_tensor)
            if class_idx is None:
                class_idx = int(out_bin.argmax(dim=1).item())
            score = out_bin[0, class_idx]
            self.model.zero_grad()
            score.backward(retain_graph=True)
            # no spatial activations available
            # return an empty heatmap
            H = input_tensor.shape[2]
            W = input_tensor.shape[3]
            return np.zeros((H,W))

        # register forward hook to capture activations
        def forward_hook(module, inp, outp):
            saved['act'] = outp
        def backward_hook(module, grad_in, grad_out):
            saved['grad'] = grad_out[0]
        h1 = target_module.register_forward_hook(forward_hook)
        h2 = target_module.register_backward_hook(backward_hook)
        hook_handles.extend([h1,h2])

        # forward
        out_bin, out_multi = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(out_bin.argmax(dim=1).item())
        score = out_bin[0, class_idx]
        # backward on score
        self.model.zero_grad()
        score.backward(retain_graph=True)
        # get activations and gradients
        act = saved['act']  # tensor
        grad = saved['grad']
        if act is None or grad is None:
            # cleanup hooks
            for hh in hook_handles:
                hh.remove()
            H = input_tensor.shape[2]
            W = input_tensor.shape[3]
            return np.zeros((H,W))
        # global-average pool grads
        weights = torch.mean(grad, dim=(2,3), keepdim=True)  # (B,C,1,1)
        cam = torch.sum(weights * act, dim=1, keepdim=True)  # (B,1,H,W)
        cam = F.relu(cam)
        cam = cam.detach().cpu().squeeze().numpy()
        # normalize
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        # remove hooks
        for hh in hook_handles:
            hh.remove()
        # resize to input resolution
        cam_resized = np.array(Image.fromarray((cam*255).astype('uint8')).resize((input_tensor.shape[3], input_tensor.shape[2]))) / 255.0
        return cam_resized

    def overlay_cam(self, img_rgb, cam, alpha=0.5):
        # img_rgb: H,W,3 in [0,1]
        heat = plt.cm.jet(cam)[:,:,:3]
        over = (1-alpha)*img_rgb + alpha*heat
        over = np.clip(over, 0, 1)
        return over

print('GradCAM utility ready')


In [ ]:
# --- Load saved model and visualize 10 random val examples with Grad-CAM ---
if not os.path.exists(SAVED_MODEL_PATH):
    raise RuntimeError(f'No saved model found at {SAVED_MODEL_PATH}. Train first or place model file there.')

# load model
model = MultiTaskSwin().to(device)
model.load_state_dict(torch.load(SAVED_MODEL_PATH, map_location=device))
model.eval()

cam_tool = GradCAM(model, device=device)

# pick 10 random samples from validation set
sample_paths = list(val_df['path'].values)
if len(sample_paths) < 10:
    sel = random.sample(sample_paths, len(sample_paths))
else:
    sel = random.sample(sample_paths, 10)

plt.figure(figsize=(16, 20))
for i, p in enumerate(sel):
    img_pil = Image.open(p).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    # predict
    with torch.no_grad():
        out_bin, out_multi = model(img_tensor)
        prob_bin = F.softmax(out_bin, dim=1).cpu().numpy()[0]
        pred_bin = int(prob_bin.argmax())
        # if predicted diseased, get multi prediction
        prob_multi = F.softmax(out_multi, dim=1).cpu().numpy()[0]
        pred_multi = int(prob_multi.argmax())
    # ground truth
    true_cls = val_df[val_df['path']==p]['class'].values[0]
    true_bin = int(val_df[val_df['path']==p]['binary'].values[0])
    # Grad-CAM on binary class index
    cam = cam_tool.compute_cam(img_tensor, class_idx=pred_bin)
    # original image for display
    img_disp = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))).astype(float)/255.0
    overlay = cam_tool.overlay_cam(img_disp, cam, alpha=0.45)
    # Plotting: original, overlay
    ax = plt.subplot(10, 2, 2*i+1)
    plt.imshow(img_disp)
    plt.title(f'True: {true_cls}\nBinary: {true_bin}')
    plt.axis('off')
    ax2 = plt.subplot(10, 2, 2*i+2)
    plt.imshow(overlay)
    # annotate predictions
    txt = f'Pred binary: {pred_bin} (p={prob_bin[pred_bin]:.2f})'
    if pred_bin==1:
        # map pred_multi back to disease name
        inv_map = {v:k for k,v in disease_map.items()}
        txt += f' | Pred multi: {inv_map[pred_multi]} (p={prob_multi[pred_multi]:.2f})'
    plt.title(txt)
    plt.axis('off')
plt.tight_layout()
plt.show()


### Notes & troubleshooting
- If Grad-CAM results appear blank, it may be that the target module selected by the heuristic does not provide spatial feature maps for your specific backbone variant. The notebook uses a heuristic to choose a target module; this works for many `timm` Swin models but you can change the target module selection logic in the `GradCAM` class.
- The final cell **always** loads `best_model.pth` and selects random examples from the validation DataFrame, so you won't retrain unnecessarily.
- To speed up visualization, reduce `IMG_SIZE` or `BATCH_SIZE`.
